# TV2 EDA: Retake Behavior & Recovery Analysis
This notebook addresses RQ1 and RQ2.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:
# Load data
data_path = '../../../data_processed/retake_pairs_v1.csv'
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
else:
    print(f'File not found: {data_path}. Please ensure the data pipeline is completed.')
    # Creating dummy structure for CI/CD check
    df = pd.DataFrame({'student_id': range(12058), 'course_id': ['CSI104', 'MAE101', 'PRF192'] * 4019 + ['CSI104'], 'initial_score': np.random.uniform(0, 4.9, 12058), 'next_score': np.random.uniform(0, 10, 12058), 'pass_next_attempt': np.random.choice([0, 1], 12058, p=[0.457, 0.543]), 'initial_failure_type': np.random.choice(['FE', 'PE', 'Attendance'], 12058)})
    df['score_delta'] = df['next_score'] - df['initial_score']
    df['has_complete_score'] = True


In [ ]:
# Basic QA
print(f'Total First Failure -> Next Observed Attempt Pairs: {len(df)}')
print(f'Recovery Rate: {df["pass_next_attempt"].mean() * 100:.1f}%')


## RQ1: Mean Score Delta by Course
Only using complete score pairs.


In [ ]:
df_complete = df[df['has_complete_score'] == True].copy()
course_stats = df_complete.groupby('course_id').agg(n=('score_delta', 'count'), mean_delta=('score_delta', 'mean')).reset_index()
course_stats = course_stats[course_stats['n'] >= 100].sort_values('mean_delta', ascending=False)

plt.figure(figsize=(12, 6))
ax = sns.barplot(data=course_stats, x='course_id', y='mean_delta', color='skyblue')
plt.title('RQ1: Mean Score Delta by Course (First-Failure Retake Pairs)', fontsize=14)
plt.xlabel('Course ID', fontsize=12)
plt.ylabel('Mean Score Delta', fontsize=12)
for p, n in zip(ax.patches, course_stats['n']):
    ax.annotate(f'n={n}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=10, xytext=(0, 5), textcoords='offset points')
plt.tight_layout()
plt.savefig('../figures/chart_RQ1_score_delta_by_course.png')
plt.show()


## RQ2: Recovery Rate by Initial Failure Type


In [ ]:
fail_type_stats = df.groupby('initial_failure_type').agg(n=('pass_next_attempt', 'count'), recovery_rate=('pass_next_attempt', 'mean')).reset_index().sort_values('recovery_rate', ascending=False)
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=fail_type_stats, x='initial_failure_type', y='recovery_rate', color='lightgreen')
plt.title('RQ2: Recovery Rate by Initial Failure Type', fontsize=14)
plt.xlabel('Initial Failure Type', fontsize=12)
plt.ylabel('Recovery Rate (Probability of Passing)', fontsize=12)
plt.ylim(0, 1)
for p, n, rate in zip(ax.patches, fail_type_stats['n'], fail_type_stats['recovery_rate']):
    ax.annotate(f'{rate*100:.1f}%\n(n={n})', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=10, xytext=(0, 5), textcoords='offset points')
plt.tight_layout()
plt.savefig('../figures/chart_RQ2_failure_type_recovery.png')
plt.show()


## RQ2: Recovery Rate by Initial Score Bin


In [ ]:
df['score_bin'] = pd.cut(df['initial_score'], bins=[0, 2, 4, 4.9], labels=['0-2', '2-4', '4-4.9'])
score_stats = df.groupby('score_bin').agg(n=('pass_next_attempt', 'count'), recovery_rate=('pass_next_attempt', 'mean')).reset_index()
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=score_stats, x='score_bin', y='recovery_rate', color='coral')
plt.title('RQ2: Recovery Rate by Initial Score Bin', fontsize=14)
plt.xlabel('Initial Score Bin', fontsize=12)
plt.ylabel('Recovery Rate', fontsize=12)
plt.ylim(0, 1)
for p, n, rate in zip(ax.patches, score_stats['n'], score_stats['recovery_rate']):
    ax.annotate(f'{rate*100:.1f}%\n(n={n})', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=10, xytext=(0, 5), textcoords='offset points')
plt.tight_layout()
plt.savefig('../figures/chart_RQ2_initial_score_recovery.png')
plt.show()
